# Extrator de metadados nos vídeos de uma OS

Autor:  Viviane da Rosa Sommer

Atualizado: 24/07/2025

Objetivo: Esse código automatiza a extração de metadados técnicos de arquivos de vídeo em um diretório, utilizando o ffprobe para coletar informações como codec, resolução, duração, taxa de bits, proporção, datas de criação/modificação e outros detalhes relevantes. Ele processa todos os vídeos encontrados, organiza os dados em um DataFrame e exporta o resultado para um arquivo CSV.

## Importações necessárias

In [1]:
import os
import subprocess
import json
import pandas as pd
import uuid
from tqdm import tqdm
from typing import Optional, Dict, Any, List

## Declaração de Constantes e Modelos

Lembre-se de atualizar as variáveis:
- folder = Path da pasta fonte dos vídeos
- output_csv = Nome do arquivo CSV que será gerado
- projeto = Nome do projeto de origem do vídeo (DESCOM,CORAL,HUB,ETC..)

In [2]:
folder = 'OS006000701300-SUB8biq25-142/videos'
output_csv = 'metadados/OS006000701300.csv'
os_name = 'OS006000701300'
projeto = 'DESCOM'

## Funções necessárias

Lembre-se de seguir o tutorial de baixar o ffprobe e atualizar a variável ffprobe_path!

In [3]:
def get_ffprobe_metadata(filepath: str) -> Optional[Dict[str, Any]]:
    """
    Executes ffprobe on the given file and returns the metadata as a dictionary.
    
    Args:
        filepath (str): Path to the video file.
    Returns:
        Optional[Dict[str, Any]]: Metadata dictionary or None if extraction fails.
    """
    ffprobe_path = 'ffmpeg-7.0.2-amd64-static/ffprobe'
    cmd = [
        ffprobe_path,
        '-v', 'quiet',
        '-print_format', 'json',
        '-show_format',
        '-show_streams',
        filepath
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout)


def parse_metadata(filepath: str) -> Optional[Dict[str, Any]]:
    """
    Parses technical metadata from a video file using ffprobe.
    
    Args:
        filepath (str): Path to the video file.
    Returns:
        Optional[Dict[str, Any]]: Dictionary with extracted metadata or None if not a valid video.
    """
    meta = get_ffprobe_metadata(filepath)
    if not meta:
        return None
    format_info = meta.get('format', {})
    streams = meta.get('streams', [])
    video_stream = next((s for s in streams if s.get('codec_type') == 'video'), None)
    if not video_stream:
        return None

    file_stats = os.stat(filepath)
    width = video_stream.get('width')
    height = video_stream.get('height')
    aspect_ratio = f"{width}:{height}" if width and height else None
    scan_type = video_stream.get('field_order', 'progressive' if video_stream.get('field_order') is None else 'interlaced')

    return {
        'id': str(uuid.uuid4()),
        'nome_arquivo': os.path.basename(filepath).split(".")[0],
        'formato_video': os.path.basename(filepath).split(".")[-1],
        'tamanho_arquivo_bytes': file_stats.st_size,
        'duracao_segundos': float(format_info.get('duration', 0)),
        'formato_container': format_info.get('format_name'),
        'taxa_bit_total': int(format_info.get('bit_rate', 0)),
        'criado_em': pd.to_datetime(file_stats.st_ctime, unit='s'),
        'modificado_em': pd.to_datetime(file_stats.st_mtime, unit='s'),
        'largura': width,
        'altura': height,
        'taxa_frames': eval(video_stream.get('r_frame_rate', '0')) if video_stream.get('r_frame_rate') else None,
        'codec': video_stream.get('codec_name'),
        'espaco_cor': video_stream.get('color_space'),
        'intervalo_cor': video_stream.get('color_range'),
        'profundidade_bits': video_stream.get('bits_per_raw_sample') or video_stream.get('bits_per_sample'),
        'tipo_varredura': scan_type,
        'tempo_inicio': float(video_stream.get('start_time', 0)),
        'perfil': video_stream.get('profile'),
        'nivel': video_stream.get('level'),
        'projeto': projeto,
        'OS': os_name
    }

## Extrair metadados

In [4]:
video_exts = ('.mp4', '.mkv', '.avi', '.mov')
files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(video_exts)]
data = []
for f in tqdm(files, desc='Processando vídeos'):
    meta = parse_metadata(f)
    if meta:
        data.append(meta)
df = pd.DataFrame(data)
df.to_csv(output_csv, index=False)

Processando vídeos: 100%|██████████| 28/28 [00:01<00:00, 15.27it/s]


In [5]:
import os
import zipfile

def zip_pasta(caminho_pasta, caminho_zip):
    # Cria o arquivo ZIP
    with zipfile.ZipFile(caminho_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Percorre todos os arquivos e subpastas na pasta
        for raiz, dirs, arquivos in os.walk(caminho_pasta):
            for arquivo in arquivos:
                # Caminho completo do arquivo
                caminho_completo = os.path.join(raiz, arquivo)
                # Adiciona o arquivo ao ZIP, mantendo a estrutura de pastas
                zipf.write(caminho_completo, os.path.relpath(caminho_completo, caminho_pasta))

# Exemplo de uso
caminho_pasta = 'metadados'
caminho_zip = 'metadados.zip'

zip_pasta(caminho_pasta, caminho_zip)

print(f"Pasta '{caminho_pasta}' compactada em '{caminho_zip}' com sucesso!")


Pasta 'metadados' compactada em 'metadados.zip' com sucesso!
